# Standard SAM optimization step
Written only; no cells executed during creation. See README.md for run order.

In [ ]:
"""Standard non-adaptive SAM with a single AdamW update per source batch."""
import math
from pathlib import Path

import torch
from torch import nn



def sam_step(model, optimizer, x, y, rho=0.05):
    """Use the same augmented batch twice; restore weights before AdamW.step()."""
    if not math.isfinite(rho) or rho < 0:
        raise ValueError('rho must be finite and nonnegative.')
    training_mode(model)
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = nn.functional.cross_entropy(logits, y)
    if not torch.isfinite(loss):
        raise RuntimeError('Non-finite SAM first-pass loss.')
    loss.backward()
    parameters = [p for p in model.parameters() if p.grad is not None]
    norm = torch.stack([p.grad.detach().norm(2) for p in parameters]).norm(2)
    if not torch.isfinite(norm):
        raise RuntimeError('Non-finite SAM gradient norm.')
    originals = []
    try:
        with torch.no_grad():
            scale = rho / norm.clamp_min(1e-12)
            for parameter in parameters:
                originals.append((parameter, parameter.detach().clone()))
                parameter.add_(parameter.grad * scale)
        optimizer.zero_grad(set_to_none=True)
        training_mode(model)  # Freeze running statistics on both passes.
        perturbed_loss = nn.functional.cross_entropy(model(x), y)
        if not torch.isfinite(perturbed_loss):
            raise RuntimeError('Non-finite SAM perturbed loss.')
        perturbed_loss.backward()
        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):
            raise RuntimeError('Non-finite SAM second-pass gradient.')
    finally:
        # Copy originals exactly, avoiding roundoff from subtracting epsilon.
        with torch.no_grad():
            for parameter, original in originals:
                parameter.copy_(original)
    optimizer.step()  # AdamW uses perturbed gradients at original parameters.
    return logits.detach(), loss.detach(), perturbed_loss.detach()


